In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
TOKEN = secrets.get_secret("HF_TOKEN")

if TOKEN:
    os.environ["HF_TOKEN"] = TOKEN

print("HF_TOKEN:", "set" if TOKEN else "MISSING (add it under Add-ons > Secrets)")

In [ ]:
%pip install -q huggingface_hub starlette-context
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
!yes n | hf download openbmb/MiniCPM5-2B-GGUF MiniCPM5-2B-Q4_K_M.gguf --local-dir /kaggle/working/models

In [ ]:
import llama_cpp
print("llama_cpp", llama_cpp.__version__)

In [ ]:
import subprocess, sys, os, time, urllib.request
!pkill -f llama_cpp.server; sleep 2
MODEL = "/kaggle/working/models/MiniCPM5-2B-Q4_K_M.gguf"
import subprocess as _sp
try:
    _smi = _sp.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                   capture_output=True, text=True, timeout=30)
    NGPUS = max(1, len([l for l in _smi.stdout.splitlines() if l.strip()]))
except Exception:
    NGPUS = 1
PORTS = [8000 + n for n in range(1, NGPUS + 1)]
print(f"detected {NGPUS} GPU(s), ports {PORTS}")
for gpu, port in enumerate(PORTS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/server{gpu}.log", "a")
    subprocess.Popen([sys.executable, "-m", "llama_cpp.server",
        "--model", MODEL, "--n_gpu_layers", "-1",
        "--host", "0.0.0.0", "--port", str(port), "--n_ctx", "8192",
        "--chat_template_kwargs", '{"enable_thinking": false}'],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=env)
    print(f"server gpu{gpu} -> port {port} launched")
for port in PORTS:
    for _ in range(60):
        time.sleep(5)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=5)
            print(f"port {port} healthy")
            break
        except Exception:
            pass
    else:
        idx = PORTS.index(port)
        print(f"PORT {port} DID NOT START - tail of server{idx}.log:")
        print(open(f"/kaggle/working/server{idx}.log").read()[-2000:])


In [ ]:
import json
import time
import urllib.request
import subprocess as _sp
try:
    _smi = _sp.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                   capture_output=True, text=True, timeout=30)
    NGPUS = max(1, len([l for l in _smi.stdout.splitlines() if l.strip()]))
except Exception:
    NGPUS = 1
PORTS = [8000 + n for n in range(1, NGPUS + 1)]
print(f"detected {NGPUS} GPU(s), ports {PORTS}")
DEADLINE = time.time() + 600
healthy = set()
while len(healthy) < len(PORTS):
    for port in PORTS:
        if port in healthy:
            continue
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=10) as resp:
                if resp.status == 200:
                    print(f"port {port} healthy")
                    healthy.add(port)
        except Exception as err:
            print(f"port {port} waiting:", err)
    if time.time() > DEADLINE:
        raise TimeoutError(f"only {sorted(healthy)} healthy in 600s; check /kaggle/working/server*.log")
    if len(healthy) < len(PORTS):
        time.sleep(10)
payload = json.dumps({"messages": [{"role": "user", "content": "Say ok."}], "temperature": 0, "max_tokens": 32}).encode()
for port in PORTS:
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        body = json.loads(resp.read().decode())
    print(f"port {port} says:", body["choices"][0]["message"]["content"])


In [ ]:
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared

import subprocess
import subprocess as _sp
try:
    _smi = _sp.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                   capture_output=True, text=True, timeout=30)
    NGPUS = max(1, len([l for l in _smi.stdout.splitlines() if l.strip()]))
except Exception:
    NGPUS = 1
PORTS = [8000 + n for n in range(1, NGPUS + 1)]
print(f"detected {NGPUS} GPU(s), ports {PORTS}")
for i, port in enumerate(PORTS):
    clog = open(f"/kaggle/working/cloudflared{i}.log", "a")
    subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=clog, stderr=subprocess.STDOUT, start_new_session=True)
print("tunnels launched, waiting for public URLs ...")
import time, re
urls = {}
for _ in range(12):
    time.sleep(5)
    for i in range(NGPUS):
        if i in urls:
            continue
        log = open(f"/kaggle/working/cloudflared{i}.log").read()
        m = re.search(r"https://[\w.-]*trycloudflare\.com", log)
        if m:
            urls[i] = m.group(0)
    if len(urls) == 2:
        break
print(urls)